# AI Tools for Actuaries
## Exercise of lecture 2: Poisson GLM in Python
### Author: Marco Maggi, Michael Mayer and Mario Wuthrich
### Version Summer School August/September 2026

In [14]:
# Import required libraries
import time
import os


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import glm

from scipy import stats
from sklearn.metrics import mean_poisson_deviance

### Load Data and split into fixed Learn and Test

In [15]:
# adapt path if needed
os.chdir('/Users/mervedosa/Documents/Repos/actuarial_deep_learning/')
df = pd.read_parquet("data/freMTPL2freq.parquet")
print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 678007 entries, 0 to 678006
Data columns (total 14 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   IDpol       678007 non-null  float64 
 1   Exposure    678007 non-null  float64 
 2   Area        678007 non-null  category
 3   VehPower    678007 non-null  int32   
 4   VehAge      678007 non-null  int32   
 5   DrivAge     678007 non-null  int32   
 6   BonusMalus  678007 non-null  int32   
 7   VehBrand    678007 non-null  category
 8   VehGas      678007 non-null  category
 9   Density     678007 non-null  int32   
 10  Region      678007 non-null  category
 11  ClaimTotal  678007 non-null  float64 
 12  ClaimNb     678007 non-null  float64 
 13  LearnTest   678007 non-null  str     
dtypes: category(4), float64(4), int32(5), str(1)
memory usage: 42.0 MB
None


,IDpol,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimTotal,ClaimNb,LearnTest
0,4156370.0,0.06,D,6,6,20,100,B2,Regular,525,R82,0.0,0.0,L
1,4006798.0,0.29,E,6,7,29,59,B12,Diesel,2498,R72,0.0,0.0,L
2,6084964.0,0.46,C,7,10,27,68,B1,Diesel,123,R82,0.0,0.0,L
3,2228865.0,0.08,D,4,15,34,50,B2,Regular,1109,R24,0.0,0.0,L
4,4141911.0,1.00,A,5,22,44,50,B3,Diesel,34,R72,0.0,0.0,L


In [16]:
# Split data first (same split and order as Wuthrich-Merz, Springer 2023)
learn_raw = df[df.LearnTest == "L"].reset_index(drop=True)
test_raw = df[df.LearnTest == "T"].reset_index(drop=True)

### Preprocess Data for GLM

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

# DrivAge: 31-40 is placed first so it becomes the GLM reference level
_driv_age_bins = [17, 20, 25, 30, 40, 50, 70, 101]
_driv_age_labels = ["18-20", "21-25", "26-30", "31-40", "41-50", "51-70", "71+"]
_driv_age_ref_first = ["31-40"] + [lev for lev in _driv_age_labels if lev != "31-40"]


# --- preprocessing functions specific for single covariates ---


def _area_encode(X):
    """Map alphabetical area codes A-F to ordinal integers 1-6."""
    return (X.squeeze().cat.codes + 1).to_frame(name="AreaGLM")


def _vehpower_clip(X):
    """Cap vehicle power at 9 and treat as categorical."""
    return pd.DataFrame({"VehPowerGLM": pd.Categorical(X.squeeze().clip(upper=9))})


def _vehage_bin(X):
    """Bin vehicle age into three groups: 0-5, 6-12, 12+."""
    return pd.DataFrame(
        {
            "VehAgeGLM": pd.cut(
                X.squeeze(), bins=[-1, 5, 12, 101], labels=["0-5", "6-12", "12+"]
            )
        }
    )


def _drivage_bin(X):
    """Bin driver age into seven groups with 31-40 as the reference level."""
    cuts = pd.cut(X.squeeze(), bins=_driv_age_bins, labels=_driv_age_labels)
    return pd.DataFrame(
        {"DrivAgeGLM": pd.Categorical(cuts, categories=_driv_age_ref_first)}
    )


def _bonusmalus_clip(X):
    """Cap bonus-malus score at 150."""
    return pd.DataFrame({"BonusMalusGLM": X.squeeze().clip(upper=150)})


def _density_log(X):
    """Log-transform population density."""
    return pd.DataFrame({"DensityGLM": np.log(X.squeeze())})


def _ft(func, names):
    """Return a FunctionTransformer with fixed output column names."""
    return FunctionTransformer(
        func=func, feature_names_out=lambda self, _: names, check_inverse=False
    )


# --- assemble the preprocessing pipeline ---

preprocessor = ColumnTransformer(
    transformers=[
        ("area", _ft(_area_encode, ["AreaGLM"]), ["Area"]),
        ("vehpower", _ft(_vehpower_clip, ["VehPowerGLM"]), ["VehPower"]),
        ("vehage", _ft(_vehage_bin, ["VehAgeGLM"]), ["VehAge"]),
        ("drivage", _ft(_drivage_bin, ["DrivAgeGLM"]), ["DrivAge"]),
        ("bonusmalus", _ft(_bonusmalus_clip, ["BonusMalusGLM"]), ["BonusMalus"]),
        ("density", _ft(_density_log, ["DensityGLM"]), ["Density"]),
        ("passthrough", "passthrough", ["VehBrand", "VehGas", "ClaimNb", "Exposure"]),
    ],
    verbose_feature_names_out=False,
)
# Fit preprocessor on learning data only
preprocessor.set_output(transform="pandas").fit(learn_raw);

In [11]:
learn = preprocessor.transform(learn_raw).copy()
test = preprocessor.transform(test_raw).copy()

print(f"Learning set size: {learn.shape[0]}")
print(f"Test set size: {test.shape[0]}")

Learning set size: 610206
Test set size: 67801


## Fit a Poisson GLM

In [12]:
# Fit GLM (Model 1)
start_time = time.time()

# Features
features = [
    "AreaGLM",
    "DrivAgeGLM",
    "VehBrand",
    "VehGas",
    "DensityGLM",
]

# Fit a Poisson GLM using the package statsmodels.
# Set `ClaimNb` as response variable and `features` as covariates.
# Use log `Exposure` as offset.
glm1_model = glm(
    "ClaimNb ~ " + " + ".join(features),
    data=learn,
    offset=np.log(learn["Exposure"]),
    family=sm.families.Poisson(),
)

glm1_results = glm1_model.fit()
print(f"Time taken: {time.time() - start_time:.2f} seconds\n")

# Display model summary
print(glm1_results.summary())

Time taken: 1.01 seconds

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               610206
Model:                            GLM   Df Residuals:                   610186
Model Family:                 Poisson   Df Model:                           19
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -98513.
Date:                Mon, 31 Aug 2026   Deviance:                   1.5138e+05
Time:                        17:02:12   Pearson chi2:                 1.02e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.004052
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercep

### Calculate Deviance Losses

In [18]:
# Get predictions
learn["GLM1"] = glm1_results.predict(learn)
test["GLM1"] = glm1_results.predict(test)

# Calculate in-sample and out-of-sample deviance using sklearn
# For better visibility we scale with 100
learn_deviance = 100 * mean_poisson_deviance(
    learn["ClaimNb"] / learn["Exposure"], learn["GLM1"], sample_weight=learn["Exposure"]
)
test_deviance = 100 * mean_poisson_deviance(
    test["ClaimNb"] / test["Exposure"], test["GLM1"], sample_weight=test["Exposure"]
)

print(f"Learning sample: {learn_deviance:.3f}")
print(f"Test sample: {test_deviance:.3f}")


Learning sample: 46.954
Test sample: 47.179


### Stepwise backward selection (drop1-analysis)

Test the significance of each variable by dropping it from the full model one at a time.

In [ ]:
full_formula = "ClaimNb ~ " + " + ".join(features)
full_deviance = glm1_results.deviance
full_df_resid = glm1_results.df_resid

drop1_results = []
# define the list of covariates to be dropped from the full model one at a time
covariates = features
for var in covariates:
    reduced_vars = [v for v in covariates if v != var]
    reduced_formula = "ClaimNb ~ " + " + ".join(reduced_vars)
    reduced_model = glm(
        reduced_formula,
        data=learn,
        offset=np.log(learn["Exposure"]),
        family=sm.families.Poisson(),
    ).fit()

    # compute the change in deviance and change in degrees of freedom
    delta_dev = reduced_model.deviance - full_deviance
    delta_df = full
    p_value = 1 - stats.chi2.cdf(delta_dev, delta_df)
    drop1_results.append(
        {
            "Variable": var,
            "Df": int(delta_df),
            "Deviance": reduced_model.deviance,
            "LRT": delta_dev,
            "Pr(>Chi)": p_value,
        }
    )

drop1_df = pd.DataFrame(drop1_results)
print("Stepwise backward selection (drop1-analysis)")
print("=" * 65)
print(f"{'Variable':<15} {'Df':>4} {'Deviance':>12} {'LRT':>12} {'Pr(>Chi)':>12}")
print("-" * 65)
print(f"{'<none>':<15} {'':>4} {full_deviance:>12.1f}")
for _, row in drop1_df.iterrows():
    p_str = (
        f"{row['Pr(>Chi)']:.4e}"
        if row["Pr(>Chi)"] < 0.001
        else f"{row['Pr(>Chi)']:.4f}"
    )
    print(
        f"{row['Variable']:<15} {row['Df']:>4} {row['Deviance']:>12.1f} {row['LRT']:>12.2f} {p_str:>12}"
    )

PatsyError: expected a noun, but instead the expression ended
    ClaimNb ~
            ^

### Stepwise forward selection (ANOVA analysis)

Test the significance of each variable by adding them sequentially to the model.

In [ ]:
anova_results = []

# Fit the null model (intercept only)
null_model = ...

prev_deviance = null_model.deviance
prev_df = null_model.df_resid

for i, var in enumerate(covariates):
    current_formula = "ClaimNb ~ " + " + ".join(covariates[: i + 1])
    current_model = glm(
        current_formula,
        data=learn,
        offset=np.log(learn["Exposure"]),
        family=sm.families.Poisson(),
    ).fit()

    # compute the change in deviance and change in degrees of freedom
    delta_dev = ...
    delta_df = ...
    p_value = 1 - stats.chi2.cdf(delta_dev, delta_df)

    anova_results.append(
        {
            "Variable": var,
            "Df": int(delta_df),
            "Deviance": current_model.deviance,
            "Delta_Dev": delta_dev,
            "Pr(>Chi)": p_value,
        }
    )

    # update the previous deviance and degrees of freedom for the next iteration
    prev_deviance = ...
    prev_df = ...

anova_df = pd.DataFrame(anova_results)
print("Stepwise forward selection (ANOVA analysis)")
print("=" * 65)
print(f"{'Variable':<15} {'Df':>4} {'Deviance':>12} {'Delta Dev':>12} {'Pr(>Chi)':>12}")
print("-" * 65)
print(f"{'NULL':<15} {'':>4} {null_model.deviance:>12.1f}")
for _, row in anova_df.iterrows():
    p_str = (
        f"{row['Pr(>Chi)']:.4e}"
        if row["Pr(>Chi)"] < 0.001
        else f"{row['Pr(>Chi)']:.4f}"
    )
    print(
        f"{row['Variable']:<15} {row['Df']:>4} {row['Deviance']:>12.1f} {row['Delta_Dev']:>12.2f} {p_str:>12}"
    )

Introduce a second GLM with different covariates for model comparison.

In [ ]:
covariates_glm_2 = [
    "DrivAgeGLM",
    "BonusMalusGLM",
    "VehBrand",
    "VehGas",
    "DensityGLM",
    "VehAgeGLM",
]
# Fit GLM Model 2 (different covariates)
glm2_model = ...

glm2_results = glm2_model.fit()
print(glm2_results.summary())

In [ ]:
# Predict in-sample and out-of-sample for Model 2
learn["GLM2"] = glm2_results.predict(learn)
test["GLM2"] = glm2_results.predict(test)

# Compare in-sample and out-of-sample losses
results = pd.DataFrame(
    {
        "Model": ["GLM 1", "GLM 2"],
        "In-sample": [
            100
            * mean_poisson_deviance(
                learn["ClaimNb"] / learn["Exposure"],
                learn["GLM1"],
                sample_weight=learn["Exposure"],
            ),
            100
            * mean_poisson_deviance(
                learn["ClaimNb"] / learn["Exposure"],
                learn["GLM2"],
                sample_weight=learn["Exposure"],
            ),
        ],
        "Out-of-sample": [
            100
            * mean_poisson_deviance(
                test["ClaimNb"] / test["Exposure"],
                test["GLM1"],
                sample_weight=test["Exposure"],
            ),
            100
            * mean_poisson_deviance(
                test["ClaimNb"] / test["Exposure"],
                test["GLM2"],
                sample_weight=test["Exposure"],
            ),
        ],
    }
)
print(results.to_string(index=False, float_format="{:.3f}".format))

### Actual vs. Predicted Plot

In [ ]:
# Bin observations by predicted frequency (GLM1)
test["glm1_claims"] = test["GLM1"] * test["Exposure"]
# Decile bins 1–10 + bin edges
test["glm1_bin"], glm1_bin_edges = pd.qcut(
    test["GLM1"],
    q=10,
    labels=False,
    retbins=True,
)
test["glm1_bin"] += 1  # make bins 1–10 instead of 0–9

# claims and exposure by bin
glm1_agg = ...

# Bin observations by predicted frequency (GLM2)
test["glm2_claims"] = test["GLM2"] * test["Exposure"]
test["glm2_bin"], glm2_bin_edges = pd.qcut(
    test["GLM2"],
    q=10,
    labels=False,
    retbins=True,
)
test["glm2_bin"] += 1

# claims and exposure by bin
glm2_agg = ...


In [ ]:
# Actual vs. Predicted plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Determine common log-frequency axis range across both models
all_bin_edges = np.concatenate(
    [glm1_bin_edges[glm1_bin_edges > 0], glm2_bin_edges[glm2_bin_edges > 0]]
)
log_freq_range = [np.log(all_bin_edges.min()), np.log(all_bin_edges.max())]

# Model 1
ax = axes[0]
log_pred1 = np.log(glm1_agg["pred_claims"] / glm1_agg["exposure"])
log_act1 = np.log(glm1_agg["actual_claims"] / glm1_agg["exposure"])
ax.plot(log_freq_range, log_freq_range, color="orange", linewidth=2)
# Draw bin rectangles
for i in range(10):
    x_left, x_right = (
        np.log(glm1_bin_edges[i]) if glm1_bin_edges[i] > 0 else log_freq_range[0],
        np.log(glm1_bin_edges[i + 1])
        if glm1_bin_edges[i + 1] > 0
        else log_freq_range[0],
    )
    y_val = log_act1.iloc[i] if i < len(log_act1) else 0
    ax.plot(
        [x_left, x_left, x_right, x_right],
        [log_freq_range[0], y_val, y_val, log_freq_range[0]],
        color="black",
        linewidth=0.8,
    )
ax.scatter(log_pred1, log_act1, color="blue", s=40, zorder=5)
ax.set_xlim(log_freq_range)
ax.set_ylim(log_freq_range)
ax.set_xlabel("estimated regression function (log-scale)")
ax.set_ylabel("bin averages of actuals (log-scale)")
ax.set_title("actual vs. predicted plot: Model 1")

# Model 2
ax = axes[1]
log_pred2 = np.log(glm2_agg["pred_claims"] / glm2_agg["exposure"])
log_act2 = np.log(glm2_agg["actual_claims"] / glm2_agg["exposure"])
ax.plot(log_freq_range, log_freq_range, color="orange", linewidth=2)
for i in range(10):
    x_left, x_right = (
        np.log(glm2_bin_edges[i]) if glm2_bin_edges[i] > 0 else log_freq_range[0],
        np.log(glm2_bin_edges[i + 1])
        if glm2_bin_edges[i + 1] > 0
        else log_freq_range[0],
    )
    y_val = log_act2.iloc[i] if i < len(log_act2) else 0
    ax.plot(
        [x_left, x_left, x_right, x_right],
        [log_freq_range[0], y_val, y_val, log_freq_range[0]],
        color="black",
        linewidth=0.8,
    )
ax.scatter(log_pred2, log_act2, color="blue", s=40, zorder=5)
ax.set_xlim(log_freq_range)
ax.set_ylim(log_freq_range)
ax.set_xlabel("estimated regression function (log-scale)")
ax.set_ylabel("bin averages of actuals (log-scale)")
ax.set_title("actual vs. predicted plot: Model 2")

plt.tight_layout()
plt.show()


### Lift Plots

In [ ]:
# Lift plots
log_act_freq1 = np.log(glm1_agg["actual_claims"] / glm1_agg["exposure"])
log_pred_freq1 = np.log(glm1_agg["pred_claims"] / glm1_agg["exposure"])
log_act_freq2 = np.log(glm2_agg["actual_claims"] / glm2_agg["exposure"])
log_pred_freq2 = np.log(glm2_agg["pred_claims"] / glm2_agg["exposure"])

y_axis_range = [
    min(
        log_act_freq1.min(),
        log_pred_freq1.min(),
        log_act_freq2.min(),
        log_pred_freq2.min(),
    ),
    max(
        log_act_freq1.max(),
        log_pred_freq1.max(),
        log_act_freq2.max(),
        log_pred_freq2.max(),
    ),
]

# add more space to y-axis limits for better visualization
y_axis_range[0] -= abs(y_axis_range[0]) * 0.05
y_axis_range[1] += abs(y_axis_range[1]) * 0.05

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bin_labels = np.arange(1, 11)

# Model 1 lift plot
ax = axes[0]
# plot the mean prediction by bin
ax.scatter([...])
# plot the actuals by bin
ax.plot([...])
ax.set_ylim(y_axis_range)
ax.set_xlabel("bin labels")
ax.set_ylabel("bin averages (log-scale)")
ax.set_title("lift plot: Model 1")
ax.axhline(y=0, color="darkgray", linewidth=0.5)
for h in np.arange(-200, 1) / 4:
    if y_axis_range[0] <= h <= y_axis_range[1]:
        ax.axhline(y=h, color="darkgray", linewidth=0.3, alpha=0.5)
ax.legend(loc="upper left")

# Model 2 lift plot
ax = axes[1]
# plot the mean prediction by bin
ax.scatter([...])
# plot the actuals by bin
ax.plot([...])
ax.set_ylim(y_axis_range)
ax.set_xlabel("bin labels")
ax.set_ylabel("bin averages (log-scale)")
ax.set_title("lift plot: Model 2")
for h in np.arange(-20, 1) / 4:
    if y_axis_range[0] <= h <= y_axis_range[1]:
        ax.axhline(y=h, color="darkgray", linewidth=0.3, alpha=0.5)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()


### Double Lift Plot

In [ ]:
# Double lift plot: bin by ratio GLM2/GLM1
test["dlp_ratio"] = ...
test["dlp_bin"], dlp_bin_edges = pd.qcut(
    test["dlp_ratio"],
    q=10,
    labels=False,
    retbins=True,
)
test["dlp_bin"] += 1  # make bins 1–10 instead of 0–9

# bin using the ratio of predicted claims from Model 2 to Model 1
dlp_agg = ...

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

# complete code for the double-lift plot
ax.scatter(
    [...],
    label="predictors Model 1",
)
ax.scatter(
    [...],
    label="predictors Model 2",
)
ax.plot(
    [...],
    label="actuals",
)

ax.set_ylim(y_axis_range)
ax.set_xlabel("bin labels")
ax.set_ylabel("bin averages (log-scale)")
ax.set_title("double lift chart: Model 2 / Model 1")
for h in np.arange(-200, 1) / 4:
    if y_axis_range[0] <= h <= y_axis_range[1]:
        ax.axhline(y=h, color="darkgray", linewidth=0.3, alpha=0.5)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()


## Murphy's score decomposition

In [ ]:
from model_diagnostics.scoring import (
    PoissonDeviance,
    decompose,
    HomogeneousExpectileScore,
)

scoring_function = PoissonDeviance()
scoring_function = HomogeneousExpectileScore(degree=1.000000001, level=0.5)
df_pred_test = pd.DataFrame({"GLM2": test["GLM2"], "GLM1": test["GLM1"]})
y_test = test["ClaimNb"] / test["Exposure"]

df = decompose(
    y_obs=y_test,
    y_pred=df_pred_test,
    scoring_function=scoring_function,
    weights=test["Exposure"],
)

# scale by 100 as done before
df[["score", "uncertainty", "miscalibration", "discrimination"]] *= 100

df.sort(["score"])